# Corpus Masterclass 2: Dictionary-Based Content Analysis and Networks

A dictionary tags words with the categories they belong to. Applying one to a corpus tells us how much of each category appears in each document. This notebook applies the MAC virtue dictionary to the State of the Union corpus, plots category trends over time, and turns the category counts into a co-occurrence network.

## Today's goal

1. Read a LIWC-format dictionary file.
2. Apply it to the State of the Union corpus for a per-speech category-count table.
3. Plot one moral-vocabulary category across two centuries.
4. Plot the remaining categories with a function called in a loop.
5. Compute an adjacency matrix: which categories co-occur across speeches.
6. Draw that adjacency as a network, with two layouts.

Dictionary: `dictionaries/macdvirtue.dic`, the Curry et al. MAC virtue dictionary.

In [ ]:
# Libraries used throughout the notebook.
import sotu
import pandas
import numpy
import re
import liwc
import matplotlib.pyplot as plt
import networkx

print('Libraries imported.')

In [ ]:
# A LIWC-format dictionary has two parts. Between two lines that contain only '%', an
# id-to-name table declares the categories. After the second '%', each line is a term
# followed by the category ids it belongs to, all tab-separated. Look at the top of the
# file to see the shape.
with open('dictionaries/macdvirtue.dic') as f:
    line_number = 0
    for line in f:
        print(line.rstrip())
        line_number = line_number + 1
        if line_number >= 14:
            break

In [ ]:
# The liwc library reads the dictionary and handles the matching rules for us, including
# the trailing '*' wildcard (a term like 'tugend*' matches tugend, tugendhaft, and so on).
# load_token_parser returns a lookup function and the list of category names.
token_categories, category_names = liwc.load_token_parser('dictionaries/macdvirtue.dic')
print(f'Categories: {category_names}')

In [ ]:
# The lookup function returns the categories a token belongs to. A token can belong to
# none, one, or several categories.
for term in ['family', 'fair', 'brave', 'reciprocate']:
    print(f'{term}: {list(token_categories(term))}')

In [ ]:
# Load the State of the Union corpus.
df = sotu.load()
print(f'{len(df)} speeches loaded.')

In [ ]:
# Tokenise one speech and count how many tokens fall into each category. re.findall
# with the word pattern is a quick tokeniser for dictionary look-up.
first_tokens = re.findall(r'\w+', df.iloc[0]['text'].lower())

first_counts = {}
for name in category_names:
    first_counts[name] = 0
for token in first_tokens:
    for name in token_categories(token):
        first_counts[name] = first_counts[name] + 1

print(f'Washington 1790, category counts:')
print(first_counts)

In [ ]:
# Repeat for every speech. Each speech contributes one row of category counts, plus its
# year, president, and token total.
rows = []
for _, speech in df.iterrows():
    tokens = re.findall(r'\w+', speech['text'].lower())
    counts = {}
    for name in category_names:
        counts[name] = 0
    for token in tokens:
        for name in token_categories(token):
            counts[name] = counts[name] + 1
    counts['year'] = speech['year']
    counts['president'] = speech['president']
    counts['total_tokens'] = len(tokens)
    rows.append(counts)

virtue_df = pandas.DataFrame(rows)
print(f'{len(virtue_df)} speeches processed.')
virtue_df[['year', 'president', 'Family', 'total_tokens']].head(3)

In [ ]:
# Long speeches use more category words simply by being longer. Divide each category
# count by the speech's token total, scaled to a rate per 1000 tokens.
for name in category_names:
    virtue_df[f'{name}_rate'] = virtue_df[name] / virtue_df['total_tokens'] * 1000

virtue_df[['year', 'president', 'Family', 'Family_rate']].head(3)

In [ ]:
# Helper. Plot one category's rate across the years.
def plot_category(category_name):
    ordered = virtue_df.sort_values('year')
    plt.figure(figsize=(13, 4))
    plt.plot(ordered['year'], ordered[f'{category_name}_rate'], marker='o', markersize=3, linewidth=0.6)
    plt.xlabel('Year')
    plt.ylabel(f'{category_name} per 1000 tokens')
    plt.title(f'MAC virtue: {category_name} in State of the Union addresses')
    plt.show()

In [ ]:
# Family across two centuries.
plot_category('Family')

In [ ]:
# Every remaining category, one call each, via a loop.
for name in ['Group', 'Reciprocity', 'Heroism', 'Deference', 'Fairness', 'Property']:
    plot_category(name)

In [ ]:
# An adjacency matrix is a square table where rows and columns are the same items, here
# the seven virtues. Subset to the seven raw-count columns and convert to a numpy array.
virtue_matrix = virtue_df[category_names].to_numpy()
print(f'Matrix shape: {virtue_matrix.shape}')

In [ ]:
# Multiply the matrix transpose by the matrix. The result is 7 by 7. Each cell is large
# when two categories tend to appear in the same speeches.
adjacency = virtue_matrix.T @ virtue_matrix
pandas.DataFrame(adjacency, index=category_names, columns=category_names).astype(int)

In [ ]:
# Build a network from the adjacency matrix. Relabel the integer nodes with category
# names and drop the self-loops created by the matrix diagonal.
graph = networkx.from_numpy_array(adjacency)
relabelling = {}
for node_index, name in enumerate(category_names):
    relabelling[node_index] = name
graph = networkx.relabel_nodes(graph, relabelling)
graph.remove_edges_from(list(networkx.selfloop_edges(graph)))
print(f'{graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges.')

In [ ]:
# Helper. Draw the network with a given layout and title. Raw co-occurrence weights run
# into the millions, so scale every edge against the largest weight: the thickest edge is
# 8 points, the rest proportional. Edge transparency keeps overlapping lines readable.
def draw_network(graph, positions, title):
    weights = []
    for u, v in graph.edges():
        weights.append(graph[u][v]['weight'])
    largest = max(weights)
    widths = []
    for weight in weights:
        widths.append(weight / largest * 8)
    plt.figure(figsize=(9, 8))
    networkx.draw(
        graph,
        pos=positions,
        with_labels=True,
        node_color='lightpink',
        node_size=2600,
        edge_color='gray',
        width=widths,
        font_size=11,
    )
    plt.title(title)
    plt.show()

In [ ]:
# A force-directed layout treats edges as springs. Connected nodes pull together;
# unconnected nodes drift apart.
spring_positions = networkx.spring_layout(graph, seed=42, weight='weight')
draw_network(graph, spring_positions, 'MAC virtues: co-occurrence (force-directed layout)')

In [ ]:
# A circular layout puts nodes evenly around a ring. Position carries no meaning here, so
# the eye reads edge thickness instead.
circular_positions = networkx.circular_layout(graph)
draw_network(graph, circular_positions, 'MAC virtues: co-occurrence (circular layout)')

## What we've covered

**Python:** file reading with `with open`; line parsing with `.strip` and `.split`; `re.findall` for tokenisation; defining functions and calling them in a loop; numpy matrix transpose `.T` and matrix multiplication `@`; networkx graphs, layouts, and drawing.

**Corpus linguistics:** LIWC-format dictionaries; dictionary-based content analysis; per-document normalisation to a rate; adjacency matrices from feature matrices; force-directed and circular network layouts.

## Read more

- **MAC virtue dictionary**: Curry, Mullins & Whitehouse (2019), [https://doi.org/10.1086/701478](https://doi.org/10.1086/701478).
- **NumPy linear algebra basics**: [https://numpy.org/doc/stable/user/quickstart.html#linear-algebra](https://numpy.org/doc/stable/user/quickstart.html#linear-algebra).
- **NetworkX tutorial**: [https://networkx.org/documentation/stable/tutorial.html](https://networkx.org/documentation/stable/tutorial.html).
- **NetworkX layouts**: [https://networkx.org/documentation/stable/reference/drawing.html#module-networkx.drawing.layout](https://networkx.org/documentation/stable/reference/drawing.html#module-networkx.drawing.layout).
- **Python re module**: [https://docs.python.org/3/library/re.html](https://docs.python.org/3/library/re.html).